# GenAI Pipeline — Patents

LLM-based screening of patents for alternative protein relevance and pillar classification.
Uses Claude with structured output via the Anthropic Python SDK.

Adapted from the publications GenAI pipeline.

### 1. Imports and Configuration

In [23]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv("../.env")

DB_PATH = "../patents_training.db"
OUTPUT_DIR = Path(".")

### 2. Data Inspection

In [4]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df())

df = con.sql("SELECT * FROM patents_raw").df()   # ← UPDATE: table name
con.close()

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nScope distribution:")
print(df["scope"].value_counts())
print(f"\nPillar distribution:")
print(df["pillar"].value_counts())
df.head()

Tables:
                 name
0  patents_embeddings
1         patents_raw

Shape: (2450, 15)

Columns: ['id', 'family_id', 'application_number', 'title', 'abstract', 'cpc', 'publication_year', 'jurisdiction', 'scope', 'pillar', 'subpillar', 'research_category', 'endproduct', 'ingredient', 'truncated']

Scope distribution:
scope
in     1275
out    1175
Name: count, dtype: int64

Pillar distribution:
pillar
PB    832
CM    172
F     171
CC    100
Name: count, dtype: int64


,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,US-11540543-B2,38969498,US17036785,Sweetened consumables comprising mogroside IV ...,Disclosed are sweetened consumables and method...,"['A23L29/37', 'A23L29/30', 'A23L33/105', 'A23L...",2023,US,out,NaN,NaN,NaN,NaN,NaN,False
1,US-20210015134-A1,38969498,US17036785,CONSUMABLES,Disclosed are sweetened consumables and method...,"['A23L27/30', 'A23L2/60', 'A23V2002/00', 'A23L...",2021,US,out,NaN,NaN,NaN,NaN,NaN,False
2,US-20170298457-A1,42829610,US15360298,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12N9/1205', 'A23C2220/206', ...",2017,US,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
3,EP-2473058-A1,42829610,EP10751643A,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12R2001/46', 'C12Y207/01006'...",2012,EP,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
4,EP-3542638-A1,44510082,EP19173302.1,NUTRITIONAL COMPOSITION,Non-medical use of at least two components sel...,"['A23L33/13', 'A61K33/04', 'A23L33/40', 'A61K3...",2019,EP,out,NaN,NaN,NaN,NaN,NaN,False


### 3. Balanced Subset Creation

Two test sets with balanced representation across scope and pillar:
- `initial_test_data`: ~14 records (6 out, 2 PB, 2 F, 2 cultivated, 2 cross-cutting)
- `test_data_100`: ~110 records (40 out, 30 PB, 15 F, 15 cultivated, 10 cross-cutting)

Adjust counts to match the actual distribution in your patents dataset.

In [64]:
def create_balanced_sample(df, n_out, n_pb, n_f, n_cultivated, n_cross, random_state=4):
    out_sample  = df[df["scope"] == "out"].sample(n=n_out,        random_state=random_state)
    pb_sample   = df[df["pillar"] == "PB"].sample(n=n_pb,         random_state=random_state)
    f_sample    = df[df["pillar"] == "F"].sample(n=n_f,           random_state=random_state)
    cult_sample = df[df["pillar"] == "CM"].sample(n=n_cultivated,  random_state=random_state)
    cross_sample= df[df["pillar"] == "CC"].sample(n=n_cross,       random_state=random_state)
    combined = pd.concat(
        [out_sample, pb_sample, f_sample, cult_sample, cross_sample], ignore_index=True
    )
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [ ]:
initial_test_data = create_balanced_sample(df, n_out=6, n_pb=2, n_f=2, n_cultivated=2, n_cross=2)
print(f"initial_test_data: {initial_test_data.shape}")

initial_test_data: (14, 15)


In [66]:
test_data_100 = create_balanced_sample(df, n_out=40, n_pb=30, n_f=15, n_cultivated=15, n_cross=10)
print(f"test_data_100: {test_data_100.shape}")
print(test_data_100[["scope", "pillar"]].value_counts())

test_data_100: (110, 15)
scope  pillar
in     PB        30
       CM        15
       F         15
       CC        10
Name: count, dtype: int64


### 4. Save Subsets to Excel

In [67]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=OUTPUT_DIR):
    path = output_dir / filename
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} records to {path}")

save_subset(initial_test_data, "initial_test_data_patents_rand4.csv")
save_subset(test_data_100, "test_data_100_patents_rand4.csv")

Saved 14 records to initial_test_data_patents_rand4.csv
Saved 110 records to test_data_100_patents_rand4.csv


In [75]:
# Read subset data from file
test_data_100 = pd.read_csv("test_data_100_patents_rand4.csv")

In [76]:
test_data_100

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,US-20220177945-A1,66290499,US17416822,METHOD FOR DETECTING AND ENUMERATING OF LOW CO...,The present document is directed to a method f...,"['C12Q1/045', 'C12Q1/06', 'C12Q1/08', 'C12N1/2...",2022,US,out,NaN,NaN,NaN,NaN,NaN,False
1,EP-3801064-A1,67441529,EP19745276.6,METHODS AND COMPOSITIONS FOR INCREASING KETONE...,A method of increasing ketone bodies in an ani...,"['A23L33/12', 'A23K50/40', 'A23L33/30', 'A23K2...",2021,EP,out,NaN,NaN,NaN,NaN,NaN,False
2,US-12403076-B2,64109846,US17277805,"Compositions comprising odorless 1,2-pentanediol",Suggested is a cosmetic or pharmaceutical or d...,"['A61Q5/12', 'A61Q15/00', 'A61K8/345', 'A61Q17...",2025,US,out,NaN,NaN,NaN,NaN,NaN,False
3,EP-4529958-A3,58073446,EP25155270.9,CONTROLLABLE TRANSCRIPTION,The present invention relates to a stable meth...,"['C12N2506/45', 'C12N2506/02', 'C12N15/113', '...",2025,EP,out,NaN,NaN,,NaN,NaN,False
4,WO-2025045207-A1,89367372,CN2024/115893,"FILLING SYSTEM, FILLING METHOD THEREFOR, AND S...","A filling system, a filling method therefor, a...","['B65B2210/06', 'B65B3/12', 'B65B55/00', 'B65B...",2025,WO,out,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,US-20260060278-A1,87845515,US19104533,ANIMAL-FREE SUBSTITUTE FOOD PRODUCTS COMPRISIN...,A method for producing a substitute food produ...,"['A23L29/06', 'A23V2002/00', 'A23V2250/54', 'A...",2026,US,in,CC,NaN,Ingredient optimisation,Cross-cutting,NaN,False
106,EP-4437913-A1,85781721,EP23165203.3,TEA PRODUCING APPARATUS AND METHOD,A tea producing apparatus and corresponding me...,"['A47J31/401', 'A47J31/4403']",2024,EP,out,NaN,NaN,NaN,NaN,NaN,False
107,US-20250287980-A1,78414261,US18702604,VEGETABLE SIDESTREAM VALORISATION,The present invention relates to a method for ...,"['A23L33/10', 'A23L33/105', 'A23L19/20', 'A23V...",2025,US,out,NaN,NaN,NaN,NaN,NaN,False
108,US-20250212923-A1,81306994,US18853157,FLAVOUR DELIVERY SYSTEM,The present invention provides a solid or semi...,"['A23L27/70', 'A23J3/227', 'A23L27/2024', 'A23...",2025,US,in,CC,NaN,Ingredient optimisation,NaN,NaN,False


### 5. Load Prompt and Select Dataset

In [77]:
PROMPT_PATH = "patent_prompt_work.md"   # ← UPDATE: path to current prompt version
version = "v4"

# ← CHANGE THIS to switch between datasets
# Options: initial_test_data (14 records), test_data_100 (110 records)
DATASET = test_data_100
DATASET_STR = "test_data_100"

In [71]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)

You are an expert in alternative protein technology and patent analysis.

Your task is to classify a patent based on its title and abstract. First, determine whether the patent concerns an invention related to alternative proteins intended for use in human food, applying the definitions, boundaries, and out of scope topics below. Provide a confidence score reflecting your certainty in the scope decision. Then, if in scope, assign the relevant alternative protein category label(s).

General Definition
Alternative proteins are plant-based, fermentation-derived, or cultivated substitutes for protein-rich animal-derived foods — including both novel ingredients (e.g. single-cell proteins, mycoprotein) and direct analogues of meat, seafood, dairy, and egg products. Substitution may be at the ingredient level (e.g. a plant protein replacing whey) or the product level (e.g. plant-based dairy replacing conventional dairy), and protein need not be explicitly stated as a goal. Patents are in scop

### 6. API Call with Structured Output

In [47]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model
# If using Opus, comment out TEMPERATURE below

MAX_TOKENS = 512         # max tokens in response
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0
RETRY_MAX_SECONDS = 90.0

REPETITIONS = 1          # number of full runs; increase to measure output variance

# ================================================================
# CHECKPOINT CONFIG
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

# ================================================================
# REASONING TOGGLE
# Keep True during prompt testing; set False for production runs.
# ================================================================
INCLUDE_REASONING = True


class _ClassificationBase(BaseModel):
    scope: Literal["in", "out"]
    confidence: int = Field(ge=1, le=7)
    plant_based: bool
    fermentation: bool
    cultivated: bool
    cross_cutting: bool

class _ClassificationWithReasoning(_ClassificationBase):
    reasoning: str

ClassificationSchema = _ClassificationWithReasoning if INCLUDE_REASONING else _ClassificationBase


client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))


def classify_patent(row: pd.Series, system_prompt: str):
    user_message = f"Title: {row['title']}\n\nAbstract: {row['abstract']}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,  #### COMMENT OUT IF USING OPUS ####
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output

### 7. Error Handling with Retry

In [18]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter

In [19]:
def classify_with_error_handling(row, system_prompt):
    pat_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_patent(row, system_prompt)
            if result is None:
                print(f"  Parse failed for {pat_id}: model returned no structured output")
                return {"id": pat_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()}
            output["id"] = pat_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pat_id}: {last_error}")
    return {"id": pat_id, "status": "api_error", "error": str(last_error)}

### 8. Checkpoint Helpers

In [20]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()

### 9. Run on Test Data

In [78]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df


Run 1 / 1
  [1/110] US-20220177945-A1
  [2/110] EP-3801064-A1
  [3/110] US-12403076-B2
  [4/110] EP-4529958-A3
  [5/110] WO-2025045207-A1
  [6/110] EP-4552507-A1
  [7/110] WO-2025088652-A1
  [8/110] WO-2025243014-A3
  [9/110] US-12439878-B2
  [10/110] US-20250332204-A1
  [11/110] DE-102024116483-A1
  [12/110] JP-2025528810-A
  [13/110] WO-2025153552-A1
  [14/110] US-20250380723-A1
  [15/110] US-20250049864-A1
  [16/110] US-20250206792-A1
  [17/110] EP-4510841-A1
  [18/110] US-12302851-B2
  [19/110] EP-4436398-A1
  [20/110] EP-4377439-A1
  [21/110] US-20250048990-A1
  [22/110] US-20250081988-A1
  [23/110] DE-102023125143-A1
  [24/110] EP-4451897-A1
  [25/110] CN-121548353-A
  [26/110] CN-122121750-A
  [27/110] JP-2026506615-A
  [28/110] CN-121772846-A
  [29/110] EP-4591714-A1
  [30/110] CN-120417784-A
  [31/110] DE-102024100931-A1
  [32/110] EP-4360462-A1
  [33/110] CN-121908951-A
  [34/110] EP-4566730-A1
  [35/110] US-20250283044-A1
  [36/110] KR-20260054052-A
  [37/110] US-2025010754

,scope_LLM,confidence_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,reasoning_LLM,id,status,run
0,out,1,False,False,False,False,This patent concerns a microbiological detecti...,US-20220177945-A1,ok,1
1,out,1,False,False,False,False,This patent concerns a nutritional composition...,EP-3801064-A1,ok,1
2,out,1,False,False,False,False,"This patent concerns a cosmetic, pharmaceutica...",US-12403076-B2,ok,1
3,out,2,False,False,False,False,While this patent on controllable transcriptio...,EP-4529958-A3,ok,1
4,out,1,False,False,False,False,This patent describes a generic filling and mi...,WO-2025045207-A1,ok,1
...,...,...,...,...,...,...,...,...,...,...
105,in,7,False,True,False,False,This patent explicitly concerns animal-free su...,US-20260060278-A1,ok,1
106,out,1,False,False,False,False,This patent concerns a tea vending apparatus a...,EP-4437913-A1,ok,1
107,out,2,False,False,False,False,This patent concerns fermentation of vegetable...,US-20250287980-A1,ok,1
108,in,6,False,False,False,True,The flavour delivery system is explicitly stat...,US-20250212923-A1,ok,1


Note: there are two routes to receiving a CC label:
1. The LLM labels it as CC directly (broad/cross-pillar AP patents).
2. The code below derives CC when more than one individual pillar flag is True (patent covers multiple AP categories).

In [79]:
# Compare LLM predictions against ground truth labels
result_cols = ["id", "run", "scope_LLM", "confidence_LLM", "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "cross_cutting_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.insert(result_cols.index("confidence_LLM") + 1, "reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "scope", "pillar"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison["correct_scope"] = comparison["scope"] == comparison["scope_LLM"]

# Derive pillar_LLM from boolean flags
pillar_flags = ["plant_based_LLM", "fermentation_LLM", "cultivated_LLM"]
comparison["pillar_LLM"] = comparison[pillar_flags + ["cross_cutting_LLM"]].apply(
    lambda r: "CC" if r[pillar_flags].sum() > 1 or (r["cross_cutting_LLM"] and r[pillar_flags].sum() == 0)
              else "PB" if r["plant_based_LLM"]
              else "F"  if r["fermentation_LLM"]
              else "CM" if r["cultivated_LLM"]
              else "NA",
    axis=1
)
comparison["correct_pillar"] = comparison["pillar"].fillna('NA') == comparison["pillar_LLM"]

# Summary metrics
print(f"Scope accuracy ({REPETITIONS} run(s)):  {comparison['correct_scope'].mean():.0%}  (n={len(comparison)})")
print(f"Pillar accuracy ({REPETITIONS} run(s)): {comparison['correct_pillar'].mean():.0%}  (n={len(comparison)})")

both_in = (comparison["scope"] == "in") & (comparison["scope_LLM"] == "in")
print(f"Pillar accuracy — both in-scope:   {comparison.loc[both_in, 'correct_pillar'].mean():.0%}  (n={both_in.sum()})")

if REPETITIONS > 1:
    per_run = comparison.groupby("run")[["correct_scope", "correct_pillar"]].mean()
    print(f"\nPer-run accuracy:\n{per_run.to_string()}")

display_cols = ["id", "title", "abstract", "scope", "scope_LLM", "confidence_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["correct_scope", "pillar", "pillar_LLM", "correct_pillar", "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "cross_cutting_LLM"]
comparison[display_cols]

Scope accuracy (1 run(s)):  96%  (n=110)
Pillar accuracy (1 run(s)): 89%  (n=110)
Pillar accuracy — both in-scope:   88%  (n=64)


,id,title,abstract,scope,scope_LLM,confidence_LLM,reasoning_LLM,correct_scope,pillar,pillar_LLM,correct_pillar,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM
0,US-20220177945-A1,METHOD FOR DETECTING AND ENUMERATING OF LOW CO...,The present document is directed to a method f...,out,out,1,This patent concerns a microbiological detecti...,True,NaN,NA,True,False,False,False,False
1,EP-3801064-A1,METHODS AND COMPOSITIONS FOR INCREASING KETONE...,A method of increasing ketone bodies in an ani...,out,out,1,This patent concerns a nutritional composition...,True,NaN,NA,True,False,False,False,False
2,US-12403076-B2,"Compositions comprising odorless 1,2-pentanediol",Suggested is a cosmetic or pharmaceutical or d...,out,out,1,"This patent concerns a cosmetic, pharmaceutica...",True,NaN,NA,True,False,False,False,False
3,EP-4529958-A3,CONTROLLABLE TRANSCRIPTION,The present invention relates to a stable meth...,out,out,2,While this patent on controllable transcriptio...,True,NaN,NA,True,False,False,False,False
4,WO-2025045207-A1,"FILLING SYSTEM, FILLING METHOD THEREFOR, AND S...","A filling system, a filling method therefor, a...",out,out,1,This patent describes a generic filling and mi...,True,NaN,NA,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,US-20260060278-A1,ANIMAL-FREE SUBSTITUTE FOOD PRODUCTS COMPRISIN...,A method for producing a substitute food produ...,in,in,7,This patent explicitly concerns animal-free su...,True,CC,F,False,False,True,False,False
106,EP-4437913-A1,TEA PRODUCING APPARATUS AND METHOD,A tea producing apparatus and corresponding me...,out,out,1,This patent concerns a tea vending apparatus a...,True,NaN,NA,True,False,False,False,False
107,US-20250287980-A1,VEGETABLE SIDESTREAM VALORISATION,The present invention relates to a method for ...,out,out,2,This patent concerns fermentation of vegetable...,True,NaN,NA,True,False,False,False,False
108,US-20250212923-A1,FLAVOUR DELIVERY SYSTEM,The present invention provides a solid or semi...,in,in,6,The flavour delivery system is explicitly stat...,True,CC,CC,True,False,False,False,True


### 10. Save to Excel for Prompt Debugging

Order of working:
1. Create a new version folder in `1_prompt_debugging/`.
2. Copy in the previous prompt, label with the new version number, make updates.
3. Edit step 10 output directory and step 5 prompt/dataset selection.
4. Run steps 5–10.
5. Manually review results.
6. Document what changed and why. Repeat from step 1.

In [80]:
save_dir = Path(version)  # CHANGE ME
save_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([
    {"metric": "scope_accuracy",                "value": f"{comparison['correct_scope'].mean():.0%}",               "n": len(comparison)},
    {"metric": "pillar_accuracy",               "value": f"{comparison['correct_pillar'].mean():.0%}",              "n": len(comparison)},
    {"metric": "pillar_accuracy_both_in_scope", "value": f"{comparison.loc[both_in, 'correct_pillar'].mean():.0%}", "n": int(both_in.sum())},
])

with pd.ExcelWriter(save_dir / f"{DATASET_STR}_results_{version}.xlsx") as writer:
    comparison[display_cols].to_excel(writer, sheet_name="comparison", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)

print(f"Saved to {save_dir / f'{DATASET_STR}_results_{version}.xlsx'}")

Saved to v4\test_data_100_results_v4.xlsx


In [ ]:
# Create dataset of only rows where LLM got scope wrong, for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["correct_scope"], "id"]
incorrect_scope_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_scope_data